In [44]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
print("All libraries imported successfully!")

All libraries imported successfully!


In [45]:
fake_df = pd.read_csv("dataset/Fake.csv")
true_df = pd.read_csv("dataset/True.csv")

print("Fake news shape:", fake_df.shape)
print("True news shape:", true_df.shape)

Fake news shape: (23481, 4)
True news shape: (21417, 4)


In [46]:
# Dataset dekhte hain
print("Fake News sample:")
print(fake_df.head(3))
print("\nTrue News sample:")
print(true_df.head(3))

Fake News sample:
                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   

                date  
0  December 31, 2017  
1  December 31, 2017  
2  December 30, 2017  

True News sample:
                                               title  \
0  As U.S. budget fight looms, Republicans flip t...   
1  U.S. military to accept transgender recruits o...   
2  Senior U.S. Republican senator: 'Let Mr. Muell...   

                                                text       subject  \
0  WASHINGTON (Reuters) - The head of a conservat...  politicsNews   
1  WASHINGTON (Reuters) 

In [47]:
# Label add karte h
fake_df['label']=0 # 0=fake
true_df['label']=1  # 1= true

print("Labels added!")
print(fake_df['label'].value_counts())
print(true_df['label'].value_counts())

Labels added!
label
0    23481
Name: count, dtype: int64
label
1    21417
Name: count, dtype: int64


In [48]:
# dono dataset combine karte h
df= pd.concat([fake_df, true_df], ignore_index=True)

print("Combine datset shape:", df.shape)
print("\nLabel count:")
print(df['label'].value_counts())

Combine datset shape: (44898, 5)

Label count:
label
0    23481
1    21417
Name: count, dtype: int64


In [49]:
# missing value check karte h
print("Missing values:")
print(df.isnull().sum())

Missing values:
title      0
text       0
subject    0
date       0
label      0
dtype: int64


In [50]:
# sirf text aur label rkege
df=df[['text','label']]
print("Dataset ready!")
print(df.shape)
print(df.head(3))

Dataset ready!
(44898, 2)
                                                text  label
0  Donald Trump just couldn t wish all Americans ...      0
1  House Intelligence Committee Chairman Devin Nu...      0
2  On Friday, it was revealed that former Milwauk...      0


In [51]:
# Text clean karte hain
import re

def clean_text(text):
    text = text.lower()           # small letters
    text = re.sub(r'\[.*?\]', '', text)  # brackets remove
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # numbers/symbols remove
    text = re.sub(r'\s+', ' ', text)   # extra spaces remove
    return text

df['text'] = df['text'].apply(clean_text)
print("Text cleaned!")
print(df['text'][0])


Text cleaned!
donald trump just couldn t wish all americans a happy new year and leave it at that instead he had to give a shout out to his enemies haters and the very dishonest fake news media the former reality show star had just one job to do and he couldn t do it as our country rapidly grows stronger and smarter i want to wish all of my friends supporters enemies haters and even the very dishonest fake news media a happy and healthy new year president angry pants tweeted will be a great year for america as our country rapidly grows stronger and smarter i want to wish all of my friends supporters enemies haters and even the very dishonest fake news media a happy and healthy new year will be a great year for america donald j trump realdonaldtrump december trump s tweet went down about as welll as you d expectwhat kind of president sends a new year s greeting like this despicable petty infantile gibberish only trump his lack of decency won t even allow him to rise above the gutter lon

In [52]:
from sklearn.model_selection import train_test_split

X= df['text']
Y= df['label']

X_train,X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.2, random_state=42)
print("Training size:",X_train.shape)
print("Training size:",X_test.shape)

Training size: (35918,)
Training size: (8980,)


In [53]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf= TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.fit_transform(X_test)

print("TF-IDF DONE!")
print("Shape:", X_train_tfidf.shape)


TF-IDF DONE!
Shape: (35918, 5000)


In [54]:
from sklearn.linear_model import LogisticRegression
model= LogisticRegression()
model.fit(X_train_tfidf,Y_train)

print("Model trained!")

Model trained!


In [55]:
from sklearn.metrics import accuracy_score, classification_report

Y_pred=model.predict(X_test_tfidf)
accuracy=accuracy_score(Y_test,Y_pred)

print("Accuracy :", accuracy*100, "%")
print("\nReport:")
print(classification_report(Y_test, Y_pred))

Accuracy : 52.628062360801785 %

Report:
              precision    recall  f1-score   support

           0       0.53      1.00      0.69      4733
           1       0.41      0.00      0.01      4247

    accuracy                           0.53      8980
   macro avg       0.47      0.50      0.35      8980
weighted avg       0.47      0.53      0.37      8980



In [56]:
print(df['label'].value_counts())
print(df.shape)

label
0    23481
1    21417
Name: count, dtype: int64
(44898, 2)


In [57]:
# Shuffle karte hain pehle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

tfidf = TfidfVectorizer(max_features=10000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

accuracy = model.score(X_test_tfidf, y_test)
print("Accuracy:", accuracy * 100, "%")

Accuracy: 98.57461024498888 %


In [58]:
def predict_news(news):
    cleaned = clean_text(news)
    vectorized = tfidf.transform([cleaned])
    prediction = model.predict(vectorized)
    if prediction[0] == 0:
        return "❌ FAKE NEWS!"
    else:
        return "✅ REAL NEWS!"

# Test karo
test_news = "Donald Trump wins the election by a landslide!"
print(predict_news(test_news))

❌ FAKE NEWS!


In [59]:
test_news = "narendra modi is prime minister of india"
print(predict_news(test_news))

❌ FAKE NEWS!


In [60]:

test_news = "donald trumpis the president of United States"
print(predict_news(test_news))

✅ REAL NEWS!


In [61]:
import pickle

# Model save karo
pickle.dump(model, open('model.pkl', 'wb'))
pickle.dump(tfidf, open('tfidf.pkl', 'wb'))

print("Model saved! ✅")

Model saved! ✅


In [62]:
import sqlite3

# Database banao
conn = sqlite3.connect('fake_news.db')
cursor = conn.cursor()

# Table banao
cursor.execute('''
    CREATE TABLE IF NOT EXISTS predictions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        news_text TEXT,
        prediction TEXT,
        date TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
''')

conn.commit()
print("Database ready! ✅")

Database ready! ✅


In [63]:
def predict_and_save(news):
    # Predict karo
    cleaned = clean_text(news)
    vectorized = tfidf.transform([cleaned])
    prediction = model.predict(vectorized)
    
    if prediction[0] == 0:
        result = "FAKE NEWS"
    else:
        result = "REAL NEWS"
    
    # SQL mein save karo
    cursor.execute('''
        INSERT INTO predictions (news_text, prediction)
        VALUES (?, ?)
    ''', (news, result))
    conn.commit()
    
    return result

# Test karo
test = "Joe Biden is the president of United States"
result = predict_and_save(test)
print("Prediction:", result)
print("Saved in database! ✅")

Prediction: FAKE NEWS
Saved in database! ✅


In [64]:
# Database se data dekhte hain
cursor.execute('SELECT * FROM predictions')
rows = cursor.fetchall()

for row in rows:
    print(row)

(1, 'Joe Biden is the president of United States', 'FAKE NEWS', '2026-03-12 06:32:15')
(2, 'us id richest country in india\n', 'FAKE NEWS', '2026-03-12 06:34:11')
(3, 'us id richest country in world\n', 'FAKE NEWS', '2026-03-12 06:34:23')
(4, 'donals trump president of usa\n\n', 'FAKE NEWS', '2026-03-12 06:35:12')
(5, 'Donald trump president of usa\n\n', 'FAKE NEWS', '2026-03-12 06:36:47')
(6, 'Donald trump president of US\n\n', 'REAL NEWS', '2026-03-12 06:36:58')
(7, 'JOE BIDEN WAS FORMER PRESIDENT OF US\n', 'FAKE NEWS', '2026-03-12 07:02:35')
(8, 'DONALD TRUMP IS\n  PRESIDENT OF US\n', 'FAKE NEWS', '2026-03-12 07:03:10')
(9, 'DONALD TRUMP IS PRESIDENT OF US\n', 'FAKE NEWS', '2026-03-12 07:03:20')
(10, 'donald trump is president of us\n\n', 'FAKE NEWS', '2026-03-12 07:03:44')
(11, 'Joe Biden is the president of United States', 'FAKE NEWS', '2026-03-12 07:14:02')
(12, 'The second term of Donald Trump\'s presidency has been characterized as a shift away from established, post-1945 inter

In [79]:
import tkinter as tk
from tkinter import messagebox

def check_news():
    news = text_box.get("1.0", tk.END)
    result = predict_and_save(news)
    if result == "FAKE NEWS":
        messagebox.showwarning("Result", "❌ FAKE NEWS!")
    else:
        messagebox.showinfo("Result", "✅ REAL NEWS!")

# Window banao
window = tk.Tk()
window.title("Fake News Detector")
window.geometry("600x400")

# Label
tk.Label(window, text="Enter News:", font=("Arial", 14)).pack(pady=10)

# Text box
text_box = tk.Text(window, height=10, width=60)
text_box.pack(pady=10)

# Button
tk.Button(window, text="Check News", 
          command=check_news,
          bg="navy", fg="white",
          font=("Arial", 12)).pack(pady=10)

window.mainloop()

In [66]:
test1 = "Joe Biden is the president of United States"
test2 = "Aliens have landed on Earth"
test3 = "NASA launches new rocket to moon"

print(predict_news(test1))
print(predict_news(test2))
print(predict_news(test3))

❌ FAKE NEWS!
❌ FAKE NEWS!
❌ FAKE NEWS!


In [67]:
test = "Earth revolves around the Sun"
print(predict_news(test))

❌ FAKE NEWS!


In [68]:
test = "Python is a programming language"
print(predict_news(test))

❌ FAKE NEWS!


In [69]:
# Real news examples
test1 = "Scientists discover new species of fish in Amazon river"
test2 = "Apple releases new iPhone with advanced camera features"  
test3 = "NASA confirms water ice found on Mars surface"
test4 = "World Health Organization declares new health guidelines"

print(predict_news(test1))
print(predict_news(test2))
print(predict_news(test3))
print(predict_news(test4))

❌ FAKE NEWS!
❌ FAKE NEWS!
❌ FAKE NEWS!
❌ FAKE NEWS!


In [73]:
test1 = "donald trump is president of us"
print(predict_news(test1))

❌ FAKE NEWS!


In [74]:
# Dataset ki news dekho
print("=== FAKE NEWS EXAMPLES ===")
print(fake_df['title'].head(5))

print("\n=== REAL NEWS EXAMPLES ===")
print(true_df['title'].head(5))

print("\n=== SUBJECTS IN FAKE NEWS ===")
print(fake_df['subject'].value_counts())

print("\n=== SUBJECTS IN REAL NEWS ===")
print(true_df['subject'].value_counts())

print("\n=== DATE RANGE ===")
print("Fake news dates:", fake_df['date'].min(), "to", fake_df['date'].max())
print("Real news dates:", true_df['date'].min(), "to", true_df['date'].max())

=== FAKE NEWS EXAMPLES ===
0     Donald Trump Sends Out Embarrassing New Year’...
1     Drunk Bragging Trump Staffer Started Russian ...
2     Sheriff David Clarke Becomes An Internet Joke...
3     Trump Is So Obsessed He Even Has Obama’s Name...
4     Pope Francis Just Called Out Donald Trump Dur...
Name: title, dtype: str

=== REAL NEWS EXAMPLES ===
0    As U.S. budget fight looms, Republicans flip t...
1    U.S. military to accept transgender recruits o...
2    Senior U.S. Republican senator: 'Let Mr. Muell...
3    FBI Russia probe helped by Australian diplomat...
4    Trump wants Postal Service to charge 'much mor...
Name: title, dtype: str

=== SUBJECTS IN FAKE NEWS ===
subject
News               9050
politics           6841
left-news          4459
Government News    1570
US_News             783
Middle-east         778
Name: count, dtype: int64

=== SUBJECTS IN REAL NEWS ===
subject
politicsNews    11272
worldnews       10145
Name: count, dtype: int64

=== DATE RANGE ===
Fake news

In [76]:
test1 = "Federal Reserve raises interest rates by 0.25 percent"
test2 = "United Nations calls for ceasefire in conflict zones"
test3 = "Scientists develop new vaccine for malaria"

print(predict_news(test1))
print(predict_news(test2))
print(predict_news(test3))

❌ FAKE NEWS!
❌ FAKE NEWS!
❌ FAKE NEWS!


In [78]:
test3 = "united state capital is washington"
print(predict_news(test3))

✅ REAL NEWS!


In [80]:
test1 = "The capital of United States is Washington "
test2 = "NASA is the space agency of United States"
test3 = "The United States has 50 states"
test4 = "New York is the largest city in United States"
test5 = "The White House is located in Washington "
print(predict_news(test1))
print(predict_news(test2))
print(predict_news(test3))
print(predict_news(test4))
print(predict_news(test5))

✅ REAL NEWS!
❌ FAKE NEWS!
✅ REAL NEWS!
✅ REAL NEWS!
✅ REAL NEWS!
